In [269]:
import os, pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [270]:
df = pd.read_csv('../data/data.csv')
stores = pd.read_csv('../data/stores.csv')
features = pd.read_csv('../data/features.csv')

In [271]:
df = df.merge(stores, how='left').merge(features, how='left')

In [272]:
def split_date(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df.Date.dt.year
    df['Month'] = df.Date.dt.month
    df['Day'] = df.Date.dt.day
    df['WeekOfYear'] = (df.Date.dt.isocalendar().week)*1.0  

In [273]:
split_date(df)

In [274]:
df = df.drop(['Date', 'Temperature','Fuel_Price', 'Type', 'MarkDown1', 'MarkDown2', 'MarkDown3',
             'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment'], axis=1)

In [275]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [276]:
X_train = train.drop(['Weekly_Sales'], axis=1)
y_train = train['Weekly_Sales']
X_test = test.drop(['Weekly_Sales'], axis=1)
y_test = test['Weekly_Sales']
binary_cols = ['IsHoliday']
numeric_cols = [col for col in df.columns if col not in binary_cols]

In [277]:
def save_numpy_array_data(file_path: str, array: np.array):
    dir_path = os.path.dirname(file_path)
    os.makedirs(dir_path, exist_ok=True)
    with open(file_path, 'wb') as file_obj:
        np.save(file_obj, array)
    
def save_object(file_path: str, obj: Pipeline) -> None:
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "wb") as file_obj:
        pickle.dump(obj, file_obj)

def get_transformation_pipeline(binary_cols: list, numeric_cols: list) -> Pipeline:
    ft = FunctionTransformer(lambda x: x.astype(int))
    sc = StandardScaler()
    preprocessor = ColumnTransformer([
            ('binary', ft, binary_cols),
            ('numeric', sc, numeric_cols)
    ])
    pipeline = Pipeline([('preprocessor', preprocessor)])
    return pipeline

def load_numpy_array_data(file_path: str) -> np.array:
    """
    load numpy array data from file
    file_path: str location of file to load
    return: np.array data loaded
    """
    with open(file_path, "rb") as file_obj:
        return np.load(file_obj)

In [278]:
pipeline = get_transformation_pipeline(binary_cols=['IsHoliday'], numeric_cols=[col for col in X_train.columns if col not in ['IsHoliday']])
transformation_object = pipeline.fit(X_train)
X_train_transformed = transformation_object.transform(X_train)
X_test_transformed = transformation_object.transform(X_test)

In [279]:
# train_arr = np.c_[X_train_transformed, np.array(y_train)]
# train_arr[0]

In [280]:
# train = load_numpy_array_data('/Users/uday/Desktop/walmart-sales-prediction/Artifacts/data_transformation/transformed_data/train.npy')
# X_train = train_arr[:, :-1]
# y_train = train_arr[:, -1]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
model = RandomForestRegressor()
model.fit(X_train_transformed, y_train)
y_pred = model.predict(X_test_transformed)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MSE: 11407962.683505364
R2 Score: 0.9781234951908829


In [288]:
from sklearn.metrics import mean_absolute_error
print("MAE:", mean_absolute_error(y_test, y_pred))

MAE: 1263.8673755639632


In [289]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train_transformed, y_train)
y_pred_lr = lr.predict(X_test_transformed)
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R2 Score:", r2_score(y_test, y_pred_lr))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))

MSE: 476354690.6464331
R2 Score: 0.08651737651283353
MAE: 14564.467102080993


In [290]:
from sklearn.ensemble import GradientBoostingRegressor
gbr = GradientBoostingRegressor()
gbr.fit(X_train_transformed, y_train)
y_pred_gbr = gbr.predict(X_test_transformed)
print("MSE:", mean_squared_error(y_test, y_pred_gbr))
print("R2 Score:", r2_score(y_test, y_pred_gbr))
print("MAE:", mean_absolute_error(y_test, y_pred_gbr))

MSE: 139158491.81248984
R2 Score: 0.7331424111539975
MAE: 7045.553725396652


In [ ]:
from sklearn.ensemble import AdaBoostRegressor
abr = AdaBoostRegressor()
abr.fit(X_train_transformed, y_train)
y_pred_abr = abr.predict(X_test_transformed)
print("MSE:", mean_squared_error(y_test, y_pred_abr))
print("R2 Score:", r2_score(y_test, y_pred_abr))
print("MAE:", mean_absolute_error(y_test, y_pred_abr))

MSE: 445737622.943898
R2 Score: 0.14523026394224448
MAE: 16941.67789508832


In [292]:
from sklearn.model_selection import RandomizedSearchCV


param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}
rf = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    cv=3,
    n_jobs=-1,
    n_iter=20
)
rf.fit(X_train_transformed, y_train)
y_pred_rf_cv = rf.predict(X_test_transformed)
print("MSE:", mean_squared_error(y_test, y_pred_rf_cv))
print("R2 Score:", r2_score(y_test, y_pred_rf_cv))
print("MAE:", mean_absolute_error(y_test, y_pred_rf_cv))

/Users/uday/Desktop/walmart-sales-prediction/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


MSE: 11350418.504706763
R2 Score: 0.9782338449123142
MAE: 1263.4339178650578


In [293]:
rf.best_params_

{'n_estimators': 100, 'min_samples_split': 2, 'max_depth': 30}